In [1]:
import numpy as np
import pandas as pd
from scipy.signal import butter, lfilter
from sklearn.preprocessing import StandardScaler
from typing import Tuple
import os

# 🆕 0. NEW IMPORT
from sklearn.model_selection import train_test_split 

# --- 1. CONFIGURATION CONSTANTS ---
FS = 1000             
LOW_CUTOFF = 20       
HIGH_CUTOFF = 450     
FILTER_ORDER = 4      

WINDOW_SIZE_MS = 200  
OVERLAP_MS = 100      

# Set the maximum number of raw samples you want per gesture class
MAX_SAMPLES_PER_CLASS = 50000 

# Set the ratio for your test set (e.g., 0.2 = 20% test data, 80% train data)
TEST_SET_RATIO = 0.2

# --- 2. CORE UTILITY FUNCTIONS (Unchanged) ---

def butter_bandpass(lowcut: float, highcut: float, fs: float, order: int=4) -> Tuple[np.ndarray, np.ndarray]:
    """Generates the Butterworth filter coefficients (b, a)."""
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return b, a

def apply_bandpass_filter(data: np.ndarray, lowcut: float, highcut: float, fs: float, order: int=4) -> np.ndarray:
    """Applies the Bandpass filter to all channels of the EMG data."""
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    filtered_data = np.zeros_like(data, dtype=np.float64)
    for i in range(data.shape[1]):
        filtered_data[:, i] = lfilter(b, a, data[:, i])
    return filtered_data

def normalize_data(data: np.ndarray) -> np.ndarray:
    """Applies Z-score (StandardScaler) normalization to the data."""
    scaler = StandardScaler()
    return scaler.fit_transform(data)

def segment_data(X: np.ndarray, y: np.ndarray, fs: float, window_size_ms: int, overlap_ms: int) -> Tuple[np.ndarray, np.ndarray]:
    """Segments the time-series data into overlapping windows."""
    window_samples = int(fs * (window_size_ms / 1000.0))
    overlap_samples = int(fs * (overlap_ms / 1000.0))
    stride = window_samples - overlap_samples
    X_segments, y_segments = [], []

    if window_samples <= 0 or stride <= 0 or window_samples > len(X):
         raise ValueError("Invalid window/overlap configuration.")

    for i in range(0, len(X) - window_samples + 1, stride):
        segment_X = X[i : i + window_samples, :]
        X_segments.append(segment_X)
        segment_y = y[i : i + window_samples]
        unique, counts = np.unique(segment_y, return_counts=True)
        y_segments.append(unique[np.argmax(counts)])

    X_final = np.array(X_segments, dtype=np.float32)
    y_final = np.array(y_segments, dtype=np.int32)
    return X_final, y_final

# --- 3. DATA LOADING AND SAMPLING FUNCTIONS (Unchanged) ---

def load_emg_data(file_path: str) -> pd.DataFrame:
    """Loads the entire dataset into a DataFrame."""
    print(f"Loading data from: {file_path}")
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Error: File not found at {file_path}. Please check the path.")
    df = pd.read_csv(file_path)
    return df

def limit_data_by_class_count(df: pd.DataFrame, max_samples_per_class: int) -> Tuple[np.ndarray, np.ndarray]:
    """Limits the number of samples for each class to create a smaller, balanced dataset."""
    label_column = 'class'
    emg_columns = [f'channel{i}' for i in range(1, 9)]
    print(f"Original total samples: {len(df)}")
    print(f"Limiting to {max_samples_per_class} samples per class...")
    
    df_limited = df.groupby(label_column).apply(
        lambda x: x.sample(min(len(x), max_samples_per_class), random_state=42)
    ).reset_index(drop=True)
    
    X_raw = df_limited[emg_columns].values
    y_raw = df_limited[label_column].values 
    print(f"New total samples after limiting: {len(df_limited)}")
    return X_raw, y_raw

# --- 4. MAIN EXECUTION (Updated with Train/Test Split) ---

if __name__ == '__main__':
    # Update this path to your local file
    LOCAL_FILE_PATH = 'EMG-data.csv' 

    try:
        # 1. Load Data
        df_full = load_emg_data(LOCAL_FILE_PATH)

        # 2. Limit and Balance Data
        X_raw, y_raw = limit_data_by_class_count(df_full, MAX_SAMPLES_PER_CLASS)
        del df_full # Free up memory

        # 3. Bandpass Filter
        print("\nApplying Bandpass Filter...")
        X_filtered = apply_bandpass_filter(X_raw, LOW_CUTOFF, HIGH_CUTOFF, FS, FILTER_ORDER)

        # 4. Normalize Data
        print("Normalizing Data (Z-score)...")
        X_normalized = normalize_data(X_filtered)

        # 5. Segment Data into Time Windows
        print(f"Segmenting data into {WINDOW_SIZE_MS}ms windows...")
        X_final, y_final = segment_data(X_normalized, y_raw, FS, WINDOW_SIZE_MS, OVERLAP_MS)
        print(f"Total segmented windows created: {len(X_final)}")
        
        # 🆕 6. SPLIT DATASET (New Step)
        print(f"Splitting data into {1-TEST_SET_RATIO:.0%} train / {TEST_SET_RATIO:.0%} test...")
        X_train, X_test, y_train, y_test = train_test_split(
            X_final, 
            y_final, 
            test_size=TEST_SET_RATIO, 
            random_state=42,  # For reproducible splits
            stratify=y_final  # Ensures class balance in both sets
        )

        print("\n--- Final Preprocessing Results ---")
        print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
        print(f"X_test shape:  {X_test.shape} | y_test shape:  {y_test.shape}")

        # 🆕 7. DELIVERABLES: Save Final Arrays (Updated)
        print("\nSaving final arrays to disk...")
        np.save('X_train.npy', X_train)
        np.save('y_train.npy', y_train)
        np.save('X_test.npy', X_test)
        np.save('y_test.npy', y_test)
        
        print("✅ Preprocessing complete. Train and test sets saved.")

    except Exception as e:
        print(f"Data pipeline failed. Error: {e}")

Loading data from: EMG-data.csv
Original total samples: 4237907
Limiting to 50000 samples per class...


C:\Users\Aman Prajapati\AppData\Local\Temp\ipykernel_9356\650522886.py:87: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_limited = df.groupby(label_column).apply(


New total samples after limiting: 363696

Applying Bandpass Filter...
Normalizing Data (Z-score)...
Segmenting data into 200ms windows...
Total segmented windows created: 3635
Splitting data into 80% train / 20% test...

--- Final Preprocessing Results ---
X_train shape: (2908, 200, 8) | y_train shape: (2908,)
X_test shape:  (727, 200, 8) | y_test shape:  (727,)

Saving final arrays to disk...
✅ Preprocessing complete. Train and test sets saved.
